# Train LogiFlow GRPO on Colab T4

This notebook runs your existing `train_grpo.py` script directly on Colab GPU.

## Steps
1. Runtime -> Change runtime type -> GPU (T4)
2. Run all cells
3. Download artifacts from `outputs/logiflow-grpo-script/artifacts`


In [ ]:
# Config
REPO_URL = "https://github.com/Roshan5105labs/crisis-logistics-env.git"
REPO_DIR = "/content/crisis-logistics-env/crisis_logistics_env"

# Training knobs (adjust as needed)
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_STEPS = 200
NUM_GENERATIONS = 2
SAMPLES_PER_TASK = 42
OUTPUT_DIR = "outputs/logiflow-grpo-script"

print({
    "repo": REPO_URL,
    "model": MODEL_NAME,
    "max_steps": MAX_STEPS,
    "num_generations": NUM_GENERATIONS,
    "samples_per_task": SAMPLES_PER_TASK,
    "output_dir": OUTPUT_DIR,
})

In [ ]:
import os
import subprocess
from pathlib import Path

root = Path("/content/crisis-logistics-env")
if not root.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(root)])
else:
    subprocess.check_call(["git", "-C", str(root), "pull", "--ff-only"])

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".[train]"])
print("Dependencies installed")


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU detected. Switch Colab runtime to T4 GPU.")


In [ ]:
# Optional: quick environment smoke checks
!python test_engine.py
!python train_and_evaluate.py


In [ ]:
# Main training run (this executes your existing train_grpo.py)
import subprocess
import sys

cmd = [
    sys.executable,
    "train_grpo.py",
    "--model-name", MODEL_NAME,
    "--max-steps", str(MAX_STEPS),
    "--num-generations", str(NUM_GENERATIONS),
    "--samples-per-task", str(SAMPLES_PER_TASK),
    "--output-dir", OUTPUT_DIR,
]
print("Running:", " ".join(cmd))
subprocess.check_call(cmd)


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

art = Path(OUTPUT_DIR) / "artifacts"
print("Artifacts dir:", art.resolve())
if not art.exists():
    raise FileNotFoundError(f"Artifacts folder not found: {art}")

print("\nGenerated files:")
for p in sorted(art.glob("*")):
    print("-", p.name)

imp = art / "improvement.json"
if imp.exists():
    print("\nImprovement:")
    print(json.dumps(json.loads(imp.read_text(encoding="utf-8")), indent=2))

curve = art / "reward_curve.png"
if curve.exists():
    print("\nReward curve:")
    display(Image(str(curve)))

summary_csv = art / "evaluation_summary.csv"
if summary_csv.exists():
    print("\nEvaluation summary table:")
    display(pd.read_csv(summary_csv))


In [ ]:
# Optional: package artifacts for quick download
import shutil
from pathlib import Path

zip_path = Path("/content/logiflow_grpo_artifacts")
shutil.make_archive(str(zip_path), "zip", root_dir=str(Path(OUTPUT_DIR) / "artifacts"))
print("Created:", str(zip_path) + ".zip")
